---
2.3 章节练习
---
【实验题】请基于本章 SimpleModel 示例代码，完成一下改造：
1. 分别编写并跑通 Eager 原生单算子执行模式；
2. 对Eager单算子执行模式 和 npugraph_ex 图模式分别采集 NPU Profiling 性能数据，导出性能 Trace 文件；
3. 对比两张在device上的耗时， 阐述npugraph_ex 图模式的优化收益。

答案参考：
Eager单算子执行模式 和 npugraph_ex 图模式的 Profiling数据采集代码示例

In [ ]:
import torch
import torch_npu
from torch_npu.profiler import profile, ProfilerActivity
from torch.profiler import tensorboard_trace_handler

# ===================== 环境校验 =====================
print("===== 环境校验信息 =====")
print(f"PyTorch version: {torch.__version__}")
print(f"torch_npu version: {torch_npu.__version__}")
print(f"NPU可用状态: {torch_npu.npu.is_available()}")
print(f"NPU设备数量: {torch_npu.npu.device_count()}")
if torch_npu.npu.is_available():
    print(f"0号NPU设备名: {torch_npu.npu.get_device_name(0)}")

# ===================== 定义模型 =====================
class SimpleModel(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, y):
        h1 = torch.add(x, y)
        h2 = torch.add(h1, x)
        h3 = torch.add(h2, y)
        return h3

# 模型迁移至NPU
model = SimpleModel().npu()

# 构造固定输入张量
x = torch.randn(2, 64).npu()
y = torch.randn(2, 64).npu()

# ===================== 模式切换开关 =====================
# mode_type = "eager"    # 原生Eager逐算子模式
mode_type = "npugraph"  # npugraph_ex图捕获回放模式

if mode_type == "npugraph":
    print("\n>>> 当前运行模式：npugraph_ex 图模式")
    run_model = torch.compile(
        model,
        backend="npugraph_ex",
        fullgraph=True,
        dynamic=False
    )
    prof_save_path = "./prof_npugraph"
else:
    print("\n>>> 当前运行模式：原生Eager单算子模式")
    run_model = model
    prof_save_path = "./prof_eager"

# ===================== Profiling性能采集 =====================
with profile(
    activities = [ProfilerActivity.CPU, ProfilerActivity.NPU],
    record_shapes = False,
    with_stack = False,
    profile_memory = True,
    schedule = torch_npu.profiler.schedule(wait = 0, active = 2, warmup = 0, repeat = 1),
    on_trace_ready = tensorboard_trace_handler(prof_save_path)
) as prof:
    out = run_model(x, y)

print(f"\nProfiling数据已导出至目录：{prof_save_path}，请使用tensorboard查看性能时序图")

profiling采集文件参考：
- [eager模式](/eager_prof.json)
- [npugraph_ex模式](/npugraph_ex_prof.json)
    profiling文件需要通过MindStudio Insight 或者Chrome Tracing 可视化

Device上的性能收益截图展示
- Eager模式上的device上的耗时展示，其中红色框为add算子

![](../images/01eager.png)

- npugraph_ex模式device上的耗时展示

![](../images/02npugraph_ex.png)